# CICIoT2023 — Rapport complet dataset + expériences (V3 / V4)

Ce notebook génère automatiquement :

- des **figures sur le dataset** avant/après consolidation et après balancing
- des **figures sur le preprocessing**
- des **figures sur l’augmentation / oversampling contrôlé**
- des **courbes d’entraînement et de validation** pour les principaux tests
- une **comparaison V3 vs V4**
- une **comparaison hiérarchique vs non hiérarchique**
- un **résumé final des meilleurs résultats**

## Ce notebook est conçu pour ton répertoire local :

```text
E:\dataset
```

et pour tes scripts/résultats situés dans :

```text
E:\dataset\processed_merged_full
```

## Résultats attendus
Les figures et tableaux seront exportés dans :

```text
E:\dataset\processed_merged_full\report_figures_pfe
```

## Remarque importante
Le notebook essaie d’utiliser **d’abord les artifacts déjà générés** (`csv`, `json`, `training_history.csv`, `test_metrics.csv`, `run_summary.json`) afin d’éviter de rescanner inutilement les gros datasets.

Quand un artifact n’existe pas, il propose un fallback basé sur le scan des fichiers bruts.


In [1]:
# =========================
# 0) Imports et configuration
# =========================
from pathlib import Path
import json
import math
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

DATASET_ROOT = Path(r"E:\dataset")
PROCESSED_ROOT = DATASET_ROOT / "processed"
PMF_ROOT = DATASET_ROOT / "processed_merged_full"

RAW_MERGED_DIR = DATASET_ROOT / "raw" / "merged_csv" / "MERGED_CSV"
MERGED_FULL_DIR = PROCESSED_ROOT / "merged_full"
FEATURE_ANALYSIS_DIR = PROCESSED_ROOT / "feature_analysis"
REPORTS_DIR = PROCESSED_ROOT / "reports"

AUG_CONTROLLED_DIR = DATASET_ROOT / "augmented" / "controlled"
BAL_V3_DIR = PMF_ROOT / "minority_balancing_v3"
BAL_V2_DIR = PMF_ROOT / "archive_legacy" / "minority_balancing_v2"

FINAL_MLP_V3_DIR = PMF_ROOT / "final_mlp_balanced_v3_results"
FINAL_MLP_DEEPER_V3_DIR = PMF_ROOT / "final_mlp_deeper_balanced_v3_results"

HIER_V3_MODELS_DIR = PMF_ROOT / "hierarchical_final_v3_models"
HIER_V4_MODELS_DIR = PMF_ROOT / "hierarchical_final_v4_models"
HIER_FINAL_INF_V2_DIR = PMF_ROOT / "hierarchical_final_inference_v2"

ARCHIVE_ROOT = PMF_ROOT / "archive_legacy"

FIG_DIR = PMF_ROOT / "report_figures_pfe"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET_ROOT:", DATASET_ROOT)
print("FIG_DIR:", FIG_DIR)


DATASET_ROOT: E:\dataset
FIG_DIR: E:\dataset\processed_merged_full\report_figures_pfe


In [2]:
# =========================
# 1) Helpers
# =========================
def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return None
    return pd.read_csv(path, **kwargs)

def read_json_safe(path):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def show_head(df, n=5):
    if df is None:
        return
    display(df.head(n))

def save_fig(name):
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", out)

def plot_bar(df, x, y, title, xlabel=None, ylabel=None, rotation=90, figsize=(14, 6), top_n=None):
    tmp = df.copy()
    if top_n is not None:
        tmp = tmp.head(top_n)
    plt.figure(figsize=figsize)
    plt.bar(tmp[x].astype(str), tmp[y].astype(float))
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.xticks(rotation=rotation, ha="right")
    plt.grid(axis="y", alpha=0.25)

def plot_line(df, x, ys, title, xlabel=None, ylabel=None, figsize=(10, 5)):
    plt.figure(figsize=figsize)
    for col in ys:
        if col in df.columns:
            plt.plot(df[x], df[col], label=col)
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or "value")
    plt.legend()
    plt.grid(alpha=0.25)

def infer_metric_value(metrics_csv, metric_name):
    df = read_csv_safe(metrics_csv)
    if df is None or "metric" not in df.columns or "value" not in df.columns:
        return None
    row = df[df["metric"] == metric_name]
    if len(row) == 0:
        return None
    return float(row["value"].iloc[0])

def collect_experiment_summary(name, path, category, version):
    path = Path(path)
    run_summary = read_json_safe(path / "run_summary.json") or {}
    test_macro_f1 = infer_metric_value(path / "test_metrics.csv", "macro_f1")
    test_accuracy = infer_metric_value(path / "test_metrics.csv", "accuracy")
    val_macro_f1 = infer_metric_value(path / "val_metrics.csv", "macro_f1")
    val_accuracy = infer_metric_value(path / "val_metrics.csv", "accuracy")
    return {
        "name": name,
        "path": str(path),
        "category": category,
        "version": version,
        "architecture": run_summary.get("architecture"),
        "loss": run_summary.get("loss"),
        "best_epoch": run_summary.get("best_epoch"),
        "best_val_macro_f1_from_summary": run_summary.get("best_val_macro_f1"),
        "val_macro_f1": val_macro_f1,
        "val_accuracy": val_accuracy,
        "test_macro_f1": test_macro_f1,
        "test_accuracy": test_accuracy,
    }


## 2) Inventaire rapide du projet

Cette section vérifie quels artifacts existent dans le répertoire de travail et servira de base à tout le reste.


In [3]:
# =========================
# 2) Inventaire rapide
# =========================
inventory = {
    "RAW_MERGED_DIR_exists": RAW_MERGED_DIR.exists(),
    "MERGED_FULL_DIR_exists": MERGED_FULL_DIR.exists(),
    "FEATURE_ANALYSIS_DIR_exists": FEATURE_ANALYSIS_DIR.exists(),
    "AUG_CONTROLLED_DIR_exists": AUG_CONTROLLED_DIR.exists(),
    "BAL_V3_DIR_exists": BAL_V3_DIR.exists(),
    "FINAL_MLP_V3_DIR_exists": FINAL_MLP_V3_DIR.exists(),
    "FINAL_MLP_DEEPER_V3_DIR_exists": FINAL_MLP_DEEPER_V3_DIR.exists(),
    "HIER_V3_MODELS_DIR_exists": HIER_V3_MODELS_DIR.exists(),
    "HIER_V4_MODELS_DIR_exists": HIER_V4_MODELS_DIR.exists(),
    "HIER_FINAL_INF_V2_DIR_exists": HIER_FINAL_INF_V2_DIR.exists(),
}
pd.DataFrame(list(inventory.items()), columns=["artifact", "exists"])


,artifact,exists
0,RAW_MERGED_DIR_exists,False
1,MERGED_FULL_DIR_exists,True
2,FEATURE_ANALYSIS_DIR_exists,True
3,AUG_CONTROLLED_DIR_exists,True
4,BAL_V3_DIR_exists,True
5,FINAL_MLP_V3_DIR_exists,True
6,FINAL_MLP_DEEPER_V3_DIR_exists,True
7,HIER_V3_MODELS_DIR_exists,True
8,HIER_V4_MODELS_DIR_exists,True
9,HIER_FINAL_INF_V2_DIR_exists,True


## 3) Dataset avant merge / pendant shards MERGED_CSV / après merge complet

### Ce que l’on veut montrer
1. **Avant consolidation complète** : comportement des fichiers `Merged01 ... Merged63`
2. **Après merge complet** : distribution globale des classes
3. **Après preprocessing** : information utile sur features, corrélations, constantes, etc.


In [4]:
# =========================
# 3A) Analyse des shards MERGED_CSV
# =========================
merged_files = sorted(RAW_MERGED_DIR.glob("Merged*.csv")) if RAW_MERGED_DIR.exists() else []
len(merged_files), merged_files[:5]


(0, [])

In [5]:
# Option rapide : on compte seulement le nombre de lignes par shard MERGED_CSV
# Cette opération est plus légère qu'un scan complet de toutes les colonnes.
raw_file_stats = []
for f in merged_files:
    try:
        n_rows = sum(1 for _ in open(f, "r", encoding="utf-8", errors="ignore")) - 1
        raw_file_stats.append({"file": f.name, "n_rows": n_rows})
    except Exception as e:
        raw_file_stats.append({"file": f.name, "n_rows": np.nan, "error": str(e)})

raw_file_stats_df = pd.DataFrame(raw_file_stats)
raw_file_stats_df.head()


""


In [6]:
if len(raw_file_stats_df) > 0:
    plot_bar(
        raw_file_stats_df.sort_values("file"),
        x="file", y="n_rows",
        title="Nombre de lignes par shard MERGED_CSV avant consolidation complète",
        xlabel="Fichier shard", ylabel="Nombre de lignes", rotation=90, figsize=(16, 6)
    )
    save_fig("01_raw_merged_shards_row_counts.png")


In [7]:
# =========================
# 3B) Distribution globale après merge complet
# =========================
class_counts_34 = read_csv_safe(FEATURE_ANALYSIS_DIR / "class_counts_34.csv")
class_counts_grouped = read_csv_safe(FEATURE_ANALYSIS_DIR / "class_counts_grouped.csv")

print("class_counts_34 shape:", None if class_counts_34 is None else class_counts_34.shape)
print("class_counts_grouped shape:", None if class_counts_grouped is None else class_counts_grouped.shape)

show_head(class_counts_34, 10)


class_counts_34 shape: (34, 2)
class_counts_grouped shape: (14, 2)


,label,count
0,DDOS-UDP_FLOOD,1964125
1,DDOS-ICMP_FLOOD,1907601
2,DOS-UDP_FLOOD,1851354
3,DDOS-SYN_FLOOD,1764669
4,DDOS-PSHACK_FLOOD,1641934
5,DDOS-TCP_FLOOD,1560699
6,DDOS-RSTFINFLOOD,1255810
7,DDOS-SYNONYMOUSIP_FLOOD,1171402
8,DOS-SYN_FLOOD,1138654
9,DOS-TCP_FLOOD,1120236


In [8]:
# Harmonisation simple des noms de colonnes pour les figures
def normalize_count_df(df):
    if df is None:
        return None
    tmp = df.copy()
    cols = {c.lower(): c for c in tmp.columns}
    label_col = None
    count_col = None
    for c in tmp.columns:
        if "label" in c.lower() or "class" in c.lower():
            label_col = c
        if "count" in c.lower() or "samples" in c.lower() or "n_" in c.lower():
            count_col = c
    if label_col is None:
        label_col = tmp.columns[0]
    if count_col is None:
        count_col = tmp.columns[1]
    tmp = tmp.rename(columns={label_col: "label", count_col: "count"})
    return tmp[["label", "count"]].sort_values("count", ascending=False)

class_counts_34_n = normalize_count_df(class_counts_34)
class_counts_grouped_n = normalize_count_df(class_counts_grouped)

show_head(class_counts_34_n, 10)


,label,count
0,DDOS-UDP_FLOOD,1964125
1,DDOS-ICMP_FLOOD,1907601
2,DOS-UDP_FLOOD,1851354
3,DDOS-SYN_FLOOD,1764669
4,DDOS-PSHACK_FLOOD,1641934
5,DDOS-TCP_FLOOD,1560699
6,DDOS-RSTFINFLOOD,1255810
7,DDOS-SYNONYMOUSIP_FLOOD,1171402
8,DOS-SYN_FLOOD,1138654
9,DOS-TCP_FLOOD,1120236


In [9]:
if class_counts_34_n is not None:
    plot_bar(
        class_counts_34_n, x="label", y="count",
        title="Distribution des classes (34 classes) après merge complet / avant balancing",
        xlabel="Classe", ylabel="Nombre d'échantillons", rotation=90, figsize=(18, 6)
    )
    save_fig("02_class_distribution_after_full_merge_34classes.png")

if class_counts_grouped_n is not None:
    plot_bar(
        class_counts_grouped_n, x="label", y="count",
        title="Distribution des familles/groupes après merge complet",
        xlabel="Famille", ylabel="Nombre d'échantillons", rotation=45, figsize=(10, 5)
    )
    save_fig("03_family_distribution_after_full_merge.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\02_class_distribution_after_full_merge_34classes.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\03_family_distribution_after_full_merge.png


## 4) État du preprocessing

Ici on documente :
- nombre de features constantes
- near-constant
- corrélations fortes
- suggestion de features à supprimer
- résumé global du preprocessing


In [10]:
# =========================
# 4) Preprocessing diagnostics
# =========================
constant_features = read_csv_safe(FEATURE_ANALYSIS_DIR / "constant_features.csv")
near_constant_features = read_csv_safe(FEATURE_ANALYSIS_DIR / "near_constant_features.csv")
high_corr_pairs = read_csv_safe(FEATURE_ANALYSIS_DIR / "high_correlation_pairs.csv")
suggested_drop = read_csv_safe(FEATURE_ANALYSIS_DIR / "suggested_drop_correlated_features.csv")
summary_metrics = read_csv_safe(FEATURE_ANALYSIS_DIR / "summary_metrics.csv")

preproc_summary = pd.DataFrame({
    "item": [
        "constant_features",
        "near_constant_features",
        "high_correlation_pairs",
        "suggested_drop_correlated_features",
    ],
    "count": [
        0 if constant_features is None else len(constant_features),
        0 if near_constant_features is None else len(near_constant_features),
        0 if high_corr_pairs is None else len(high_corr_pairs),
        0 if suggested_drop is None else len(suggested_drop),
    ]
})
preproc_summary


,item,count
0,constant_features,0
1,near_constant_features,1
2,high_correlation_pairs,8
3,suggested_drop_correlated_features,6


In [11]:
plot_bar(
    preproc_summary,
    x="item", y="count",
    title="Résumé du preprocessing : constantes, near-constant et corrélations",
    xlabel="Type d'analyse", ylabel="Nombre", rotation=15, figsize=(10, 5)
)
save_fig("04_preprocessing_summary_counts.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\04_preprocessing_summary_counts.png


In [12]:
show_head(summary_metrics, 20)
show_head(suggested_drop, 20)


,metric,value
0,samples,2.100562e+07
1,features_total_before_drop,4.100000e+01
2,features_total_after_drop,3.600000e+01
3,final_feature_count,3.200000e+01
4,classes_34,3.400000e+01
5,classes_grouped,1.400000e+01
6,imbalance_ratio_34,1.642245e+03
7,imbalance_ratio_grouped,5.477714e+02
8,global_duplicates_removed,1.330849e+07
9,constant_features_count,0.000000e+00


,feature
0,syn_count
1,fin_count
2,rst_count
3,IPv
4,LLC
5,Tot size


## 5) Évolution du dataset après augmentation / oversampling contrôlé

Cette section compare :
- la distribution **avant balancing**
- la distribution **après balancing v3**
- la quantité de données synthétiques générées
- la plausibilité statistique des données synthétiques


In [13]:
# =========================
# 5) Balancing / augmentation
# =========================
class_before_after_v3 = read_csv_safe(BAL_V3_DIR / "class_distribution_before_after_v3.csv")
augmentation_report_v3 = read_csv_safe(BAL_V3_DIR / "augmentation_report_v3.csv")
plausibility_stats_v3 = read_csv_safe(BAL_V3_DIR / "plausibility_stats_v3.csv")
plausibility_summary_v3 = read_csv_safe(BAL_V3_DIR / "plausibility_summary_v3.csv")
balancing_summary_v3 = read_json_safe(BAL_V3_DIR / "balancing_summary_v3.json")

show_head(class_before_after_v3, 10)
show_head(augmentation_report_v3, 10)


,label,before,after
0,BACKDOOR_MALWARE,3075,300000
1,MITM-ARPSPOOFING,268962,300000
2,DOS-SYN_FLOOD,1133753,300000
3,DOS-TCP_FLOOD,1112260,300000
4,DOS-UDP_FLOOD,1720950,300000
5,MIRAI-GREETH_FLOOD,895892,300000
6,MIRAI-GREIP_FLOOD,693641,300000
7,MIRAI-UDPPLAIN,764963,300000
8,RECON-HOSTDISCOVERY,127863,300000
9,BENIGN,1047297,300000


,label,original_count,generated_count,downsampled_count,final_target
0,UPLOADING_ATTACK,1196,298804,0,300000
1,RECON-PINGSWEEP,2161,297839,0,300000
2,BACKDOOR_MALWARE,3075,296925,0,300000
3,XSS,3705,296295,0,300000
4,SQLINJECTION,5022,294978,0,300000
5,COMMANDINJECTION,5150,294850,0,300000
6,BROWSERHIJACKING,5560,294440,0,300000
7,DICTIONARYBRUTEFORCE,12520,287480,0,300000
8,DDOS-SLOWLORIS,22400,277600,0,300000
9,DDOS-HTTP_FLOOD,27597,272403,0,300000


In [14]:
# Figure avant/après balancing v3
if class_before_after_v3 is not None:
    tmp = class_before_after_v3.copy()
    cols_lower = {c.lower(): c for c in tmp.columns}

    # Essaie de détecter automatiquement les bonnes colonnes
    label_col = next((c for c in tmp.columns if "label" in c.lower() or "class" in c.lower()), tmp.columns[0])
    before_col = next((c for c in tmp.columns if "before" in c.lower()), None)
    after_col = next((c for c in tmp.columns if "after" in c.lower()), None)

    if before_col and after_col:
        tmp = tmp.rename(columns={label_col: "label", before_col: "before", after_col: "after"})
        tmp = tmp.sort_values("before", ascending=False)

        x = np.arange(len(tmp))
        width = 0.38
        plt.figure(figsize=(18, 6))
        plt.bar(x - width/2, tmp["before"], width=width, label="Before")
        plt.bar(x + width/2, tmp["after"], width=width, label="After")
        plt.xticks(x, tmp["label"], rotation=90)
        plt.title("Distribution des classes avant / après balancing v3")
        plt.xlabel("Classe")
        plt.ylabel("Nombre d'échantillons")
        plt.legend()
        plt.grid(axis="y", alpha=0.25)
        save_fig("05_class_distribution_before_after_balancing_v3.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\05_class_distribution_before_after_balancing_v3.png


In [15]:
# Figure quantité de données synthétiques par classe
if augmentation_report_v3 is not None:
    tmp = augmentation_report_v3.copy()
    label_col = next((c for c in tmp.columns if "label" in c.lower() or "class" in c.lower()), tmp.columns[0])
    gen_col = next((c for c in tmp.columns if "generated" in c.lower()), None)

    if gen_col:
        tmp = tmp.rename(columns={label_col: "label", gen_col: "generated"})
        tmp = tmp.sort_values("generated", ascending=False)
        plot_bar(
            tmp, x="label", y="generated",
            title="Quantité de données synthétiques générées par classe (balancing v3)",
            xlabel="Classe", ylabel="Nombre généré", rotation=90, figsize=(18, 6)
        )
        save_fig("06_generated_synthetic_samples_per_class_v3.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\06_generated_synthetic_samples_per_class_v3.png


In [16]:
# Figure de plausibilité (si disponible)
if plausibility_stats_v3 is not None and len(plausibility_stats_v3.columns) >= 2:
    num_cols = [c for c in plausibility_stats_v3.columns if pd.api.types.is_numeric_dtype(plausibility_stats_v3[c])]
    if len(num_cols) > 0:
        tmp = plausibility_stats_v3.copy()
        tmp = tmp.sort_values(num_cols[0], ascending=False).head(20)
        plot_bar(
            tmp.assign(feature=tmp.iloc[:, 0].astype(str)),
            x="feature", y=num_cols[0],
            title=f"Plausibility stats v3 — top 20 selon {num_cols[0]}",
            xlabel="Feature", ylabel=num_cols[0], rotation=90, figsize=(16, 6)
        )
        save_fig("07_plausibility_stats_v3_top20.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\07_plausibility_stats_v3_top20.png


## 6) Comparaison des expériences non hiérarchiques (flat)

On compare ici les principaux tests flat :
- `stable_flat_mlp_a_results`
- `final_mlp_balanced_v3_results`
- `final_mlp_deeper_balanced_v3_results`
- éventuellement d’autres résultats archivés si présents


In [17]:
# =========================
# 6) Résumés flat / non hiérarchiques
# =========================
flat_experiments = [
    collect_experiment_summary("stable_flat_mlp_a", ARCHIVE_ROOT / "stable_flat_mlp_a_results", "flat", "pre_v3"),
    collect_experiment_summary("mlp_a_pytorch", ARCHIVE_ROOT / "mlp_a_pytorch_results", "flat", "pre_v3"),
    collect_experiment_summary("mlp_b_pytorch", ARCHIVE_ROOT / "mlp_b_pytorch_results", "flat", "pre_v3"),
    collect_experiment_summary("final_mlp_balanced_v3", FINAL_MLP_V3_DIR, "flat", "v3"),
    collect_experiment_summary("final_mlp_deeper_balanced_v3", FINAL_MLP_DEEPER_V3_DIR, "flat", "v3"),
]
flat_summary_df = pd.DataFrame(flat_experiments)
flat_summary_df


,name,path,category,version,architecture,loss,best_epoch,best_val_macro_f1_from_summary,val_macro_f1,val_accuracy,test_macro_f1,test_accuracy
0,stable_flat_mlp_a,E:\dataset\processed_merged_full\archive_legac...,flat,pre_v3,32 -> 64 -> 34,None,11,0.562282,0.562282,0.744678,0.563490,0.745194
1,mlp_a_pytorch,E:\dataset\processed_merged_full\archive_legac...,flat,pre_v3,32 -> 64 -> 34,None,11,0.518029,0.518029,0.661048,0.518522,0.661490
2,mlp_b_pytorch,E:\dataset\processed_merged_full\archive_legac...,flat,pre_v3,32 -> 128 -> 64 -> 34,None,22,0.516251,0.516251,0.653381,0.516837,0.653548
3,final_mlp_balanced_v3,E:\dataset\processed_merged_full\final_mlp_bal...,flat,v3,32 -> 64 -> 34,None,74,0.556359,0.556359,0.569923,0.556701,0.570185
4,final_mlp_deeper_balanced_v3,E:\dataset\processed_merged_full\final_mlp_dee...,flat,v3,32 -> 128 -> 64 -> 34,None,64,0.498562,0.498562,0.522653,0.499024,0.522968


In [18]:
flat_summary_df_sorted = flat_summary_df.sort_values("test_macro_f1", ascending=False)
plot_bar(
    flat_summary_df_sorted,
    x="name", y="test_macro_f1",
    title="Comparaison des macro-F1 test — modèles non hiérarchiques",
    xlabel="Expérience", ylabel="Test macro-F1", rotation=25, figsize=(12, 5)
)
save_fig("08_flat_models_test_macro_f1_comparison.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\08_flat_models_test_macro_f1_comparison.png


## 7) Comparaison des expériences hiérarchiques (V3 / V4)

On résume :
- `level1_binary`
- `level2_family`
- `level3` par famille
- résultat final du pipeline hiérarchique V3
- résultats V4 lorsque disponibles


In [19]:
# =========================
# 7) Résumés hiérarchiques
# =========================
hier_experiments = []

# V3 level1 / level2
hier_experiments.append(collect_experiment_summary("hier_v3_level1_binary", HIER_V3_MODELS_DIR / "level1_binary", "hierarchical", "v3"))
hier_experiments.append(collect_experiment_summary("hier_v3_level2_family", HIER_V3_MODELS_DIR / "level2_family", "hierarchical", "v3"))

# V3 level3 families
for fam in ["ddos", "dos", "mirai", "recon", "spoofing", "web"]:
    hier_experiments.append(collect_experiment_summary(f"hier_v3_level3_{fam}", HIER_V3_MODELS_DIR / "level3_family_submodels" / fam, "hierarchical", "v3"))

# V4 level1 / level2
hier_experiments.append(collect_experiment_summary("hier_v4_level1_binary", HIER_V4_MODELS_DIR / "level1_binary", "hierarchical", "v4"))
hier_experiments.append(collect_experiment_summary("hier_v4_level2_family", HIER_V4_MODELS_DIR / "level2_family", "hierarchical", "v4"))

# V4 level3 families
for fam in ["ddos", "dos", "mirai", "recon", "spoofing", "web"]:
    hier_experiments.append(collect_experiment_summary(f"hier_v4_level3_{fam}", HIER_V4_MODELS_DIR / "level3_family_submodels" / fam, "hierarchical", "v4"))

hier_summary_df = pd.DataFrame(hier_experiments)
hier_summary_df


,name,path,category,version,architecture,loss,best_epoch,best_val_macro_f1_from_summary,val_macro_f1,val_accuracy,test_macro_f1,test_accuracy
0,hier_v3_level1_binary,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,32 -> 64 -> 2,None,7,0.541441,0.541441,0.970156,0.543347,0.970259
1,hier_v3_level2_family,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,32 -> 64 -> family_count,None,60,0.531289,0.531289,0.719921,0.530652,0.720007
2,hier_v3_level3_ddos,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,48,0.901715,0.901715,0.905185,0.901507,0.905019
3,hier_v3_level3_dos,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,60,0.980931,0.980931,0.980956,0.980552,0.980583
4,hier_v3_level3_mirai,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,54,0.987334,0.987334,0.987341,0.987734,0.987741
5,hier_v3_level3_recon,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,59,0.454662,0.454662,0.477396,0.450976,0.474293
6,hier_v3_level3_spoofing,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,53,0.701054,0.701054,0.704433,0.701311,0.704956
7,hier_v3_level3_web,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,58,0.549625,0.549625,0.550609,0.551673,0.553080
8,hier_v4_level1_binary,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v4,32 -> 128 -> 64 -> 2,focal_gamma_2.0,39,0.696111,0.696111,0.960999,0.696427,0.960901
9,hier_v4_level2_family,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v4,32 -> 128 -> 64 -> family_count,focal_gamma_2.0,65,0.556502,0.556502,0.744885,0.555822,0.744560


In [20]:
# Comparaison V3 vs V4 sur les principales briques hiérarchiques
main_hier = hier_summary_df[hier_summary_df["name"].isin([
    "hier_v3_level1_binary", "hier_v3_level2_family",
    "hier_v4_level1_binary", "hier_v4_level2_family",
    "hier_v3_level3_web", "hier_v4_level3_web",
    "hier_v3_level3_recon", "hier_v4_level3_recon",
    "hier_v3_level3_spoofing", "hier_v4_level3_spoofing",
])].copy()

main_hier


,name,path,category,version,architecture,loss,best_epoch,best_val_macro_f1_from_summary,val_macro_f1,val_accuracy,test_macro_f1,test_accuracy
0,hier_v3_level1_binary,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,32 -> 64 -> 2,None,7,0.541441,0.541441,0.970156,0.543347,0.970259
1,hier_v3_level2_family,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,32 -> 64 -> family_count,None,60,0.531289,0.531289,0.719921,0.530652,0.720007
5,hier_v3_level3_recon,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,59,0.454662,0.454662,0.477396,0.450976,0.474293
6,hier_v3_level3_spoofing,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,53,0.701054,0.701054,0.704433,0.701311,0.704956
7,hier_v3_level3_web,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,58,0.549625,0.549625,0.550609,0.551673,0.553080
8,hier_v4_level1_binary,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v4,32 -> 128 -> 64 -> 2,focal_gamma_2.0,39,0.696111,0.696111,0.960999,0.696427,0.960901
9,hier_v4_level2_family,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v4,32 -> 128 -> 64 -> family_count,focal_gamma_2.0,65,0.556502,0.556502,0.744885,0.555822,0.744560
13,hier_v4_level3_recon,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v4,32 -> 128 -> 64 -> 5,focal_gamma_2.0,37,0.465381,0.465381,0.500809,0.463275,0.499022
14,hier_v4_level3_spoofing,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v4,32 -> 128 -> 64 -> 2,focal_gamma_2.0,21,0.683158,0.683158,0.686411,0.681228,0.684689
15,hier_v4_level3_web,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v4,32 -> 128 -> 64 -> 5,focal_gamma_2.0,69,0.589781,0.589781,0.595796,0.587944,0.594200


In [21]:
if len(main_hier) > 0:
    plot_bar(
        main_hier.sort_values("test_macro_f1", ascending=False),
        x="name", y="test_macro_f1",
        title="Comparaison hiérarchique V3 vs V4 — test macro-F1",
        xlabel="Bloc", ylabel="Test macro-F1", rotation=35, figsize=(14, 6)
    )
    save_fig("09_hierarchical_v3_v4_test_macro_f1_comparison.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\09_hierarchical_v3_v4_test_macro_f1_comparison.png


## 8) Résultat final du pipeline hiérarchique

Cette section lit directement :
- `hierarchical_final_inference_v2/summary.json`
- `hierarchical_final_inference_v2/level_metrics.csv`
- `hierarchical_final_inference_v2/classification_report_final.csv`

pour documenter le meilleur pipeline hiérarchique évalué de bout en bout.


In [22]:
# =========================
# 8) Pipeline hiérarchique final
# =========================
hier_final_summary = read_json_safe(HIER_FINAL_INF_V2_DIR / "summary.json")
hier_level_metrics = read_csv_safe(HIER_FINAL_INF_V2_DIR / "level_metrics.csv")
hier_final_report = read_csv_safe(HIER_FINAL_INF_V2_DIR / "classification_report_final.csv")

hier_final_summary, show_head(hier_level_metrics, 20), show_head(hier_final_report, 10)


,level,metric,value
0,final_34class,accuracy,0.562963
1,final_34class,macro_f1,0.536215
2,final_34class,weighted_f1,0.536214
3,final_34class,macro_precision,0.576005
4,final_34class,macro_recall,0.562963
5,final_34class,balanced_accuracy,0.562963
6,level1_binary,accuracy,0.970275
7,level1_binary,macro_f1,0.543125
8,level1_binary,weighted_f1,0.958901
9,level1_binary,macro_precision,0.714691


,Unnamed: 0,precision,recall,f1-score,support
0,BACKDOOR_MALWARE,0.683155,0.056778,0.104842,45000.0
1,BENIGN,0.457219,0.057000,0.101363,45000.0
2,BROWSERHIJACKING,0.376726,0.397622,0.386892,45000.0
3,COMMANDINJECTION,0.353062,0.405511,0.377473,45000.0
4,DDOS-ACK_FRAGMENTATION,0.983082,0.981400,0.982240,45000.0
5,DDOS-HTTP_FLOOD,0.684553,0.822711,0.747300,45000.0
6,DDOS-ICMP_FLOOD,0.998683,0.994356,0.996515,45000.0
7,DDOS-ICMP_FRAGMENTATION,0.995779,0.975111,0.985337,45000.0
8,DDOS-PSHACK_FLOOD,0.911976,0.988622,0.948753,45000.0
9,DDOS-RSTFINFLOOD,0.993635,0.978244,0.985880,45000.0


({'input_csv': 'E:\\dataset\\processed_merged_full\\minority_balancing_v3\\training_ready\\test.csv',
  'output_dir': 'E:\\dataset\\processed_merged_full\\hierarchical_final_inference_v2',
  'n_rows': 1530001,
  'device': 'cuda',
  'prediction_family_distribution': {'ddos': 653689,
   'spoofing': 77446,
   'web': 307913,
   'dos': 66583,
   'mirai': 133880,
   'recon': 281138,
   'malware': 3740,
   'benign': 5610,
   'bruteforce': 2},
  'prediction_label_distribution_top20': {'DDOS-TCP_FLOOD': 87221,
   'DDOS-SYNONYMOUSIP_FLOOD': 84280,
   'UPLOADING_ATTACK': 82851,
   'VULNERABILITYSCAN': 72987,
   'RECON-HOSTDISCOVERY': 72579,
   'XSS': 67904,
   'DDOS-UDP_FLOOD': 62632,
   'SQLINJECTION': 57977,
   'RECON-PINGSWEEP': 56001,
   'DDOS-HTTP_FLOOD': 54082,
   'COMMANDINJECTION': 51685,
   'DDOS-SLOWLORIS': 50911,
   'DDOS-PSHACK_FLOOD': 48782,
   'RECON-PORTSCAN': 47681,
   'BROWSERHIJACKING': 47496,
   'DDOS-ACK_FRAGMENTATION': 44923,
   'DDOS-ICMP_FLOOD': 44805,
   'MIRAI-GREIP_FLOOD

In [23]:
if hier_level_metrics is not None:
    pivot = hier_level_metrics.pivot(index="level", columns="metric", values="value").reset_index()
    display(pivot)

    if "macro_f1" in pivot.columns:
        plot_bar(
            pivot.sort_values("macro_f1", ascending=False),
            x="level", y="macro_f1",
            title="Macro-F1 par niveau du pipeline hiérarchique final",
            xlabel="Niveau", ylabel="Macro-F1", rotation=10, figsize=(10, 5)
        )
        save_fig("10_hierarchical_final_macro_f1_by_level.png")


metric,level,accuracy,balanced_accuracy,macro_f1,macro_precision,macro_recall,weighted_f1
0,final_34class,0.562963,0.562963,0.536215,0.576005,0.562963,0.536214
1,level1_binary,0.970275,0.527475,0.543125,0.714691,0.527475,0.958901
2,level2_family,0.719136,0.528057,0.471917,0.565711,0.469384,0.686484


Saved: E:\dataset\processed_merged_full\report_figures_pfe\10_hierarchical_final_macro_f1_by_level.png


## 9) Courbes d’apprentissage (train / validation / loss)

Cette section génère automatiquement les courbes de :
- loss
- val macro-F1
- learning rate si disponible

pour les principaux tests :
- `final_mlp_balanced_v3`
- `final_mlp_deeper_balanced_v3`
- `hierarchical V3`
- `hierarchical V4`


In [24]:
# =========================
# 9) Courbes training / validation
# =========================
history_targets = {
    "flat_v3_balanced": FINAL_MLP_V3_DIR / "training_history.csv",
    "flat_v3_deeper": FINAL_MLP_DEEPER_V3_DIR / "training_history.csv",
    "hier_v3_level1": HIER_V3_MODELS_DIR / "level1_binary" / "training_history.csv",
    "hier_v3_level2": HIER_V3_MODELS_DIR / "level2_family" / "training_history.csv",
    "hier_v4_level1": HIER_V4_MODELS_DIR / "level1_binary" / "training_history.csv",
    "hier_v4_level2": HIER_V4_MODELS_DIR / "level2_family" / "training_history.csv",
    "hier_v4_web": HIER_V4_MODELS_DIR / "level3_family_submodels" / "web" / "training_history.csv",
    "hier_v4_recon": HIER_V4_MODELS_DIR / "level3_family_submodels" / "recon" / "training_history.csv",
    "hier_v4_spoofing": HIER_V4_MODELS_DIR / "level3_family_submodels" / "spoofing" / "training_history.csv",
}
history_data = {k: read_csv_safe(v) for k, v in history_targets.items()}
{k: None if v is None else v.shape for k, v in history_data.items()}


{'flat_v3_balanced': (80, 11),
 'flat_v3_deeper': (78, 11),
 'hier_v3_level1': (15, 10),
 'hier_v3_level2': (60, 10),
 'hier_v4_level1': (40, 11),
 'hier_v4_level2': (70, 10),
 'hier_v4_web': (70, 10),
 'hier_v4_recon': (47, 10),
 'hier_v4_spoofing': (31, 10)}

In [25]:
# Génère une figure par expérience : train_loss / val_loss / val_macro_f1
for exp_name, dfh in history_data.items():
    if dfh is None or "epoch" not in dfh.columns:
        continue

    cols_present = set(dfh.columns)

    if {"train_loss", "val_loss"}.intersection(cols_present):
        ys = [c for c in ["train_loss", "val_loss"] if c in dfh.columns]
        plot_line(dfh, "epoch", ys, title=f"{exp_name} — Loss curves", xlabel="Epoch", ylabel="Loss", figsize=(10, 4))
        save_fig(f"curve_{exp_name}_loss.png")

    if "val_macro_f1" in dfh.columns:
        plot_line(dfh, "epoch", ["val_macro_f1"], title=f"{exp_name} — Validation Macro-F1", xlabel="Epoch", ylabel="Macro-F1", figsize=(10, 4))
        save_fig(f"curve_{exp_name}_val_macro_f1.png")

    if "learning_rate" in dfh.columns:
        plot_line(dfh, "epoch", ["learning_rate"], title=f"{exp_name} — Learning rate", xlabel="Epoch", ylabel="LR", figsize=(10, 4))
        save_fig(f"curve_{exp_name}_learning_rate.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_flat_v3_balanced_loss.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_flat_v3_balanced_val_macro_f1.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_flat_v3_balanced_learning_rate.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_flat_v3_deeper_loss.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_flat_v3_deeper_val_macro_f1.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_flat_v3_deeper_learning_rate.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_hier_v3_level1_loss.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_hier_v3_level1_val_macro_f1.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_hier_v3_level1_learning_rate.png
Saved: E:\dataset\processed_merged_full\report_figures_pfe\curve_hier_v3_level2_loss.png
Saved: E:\dataset\processed_merged_full\report_figure

## 10) Comparaison directe V3 vs V4 sur les blocs clés

Ici on compare spécifiquement les blocs qui ont changé entre V3 et V4 :
- level1 binary
- level2 family
- web
- recon
- spoofing


In [26]:
compare_pairs = []
for block in ["level1_binary", "level2_family"]:
    v3_path = HIER_V3_MODELS_DIR / block
    v4_path = HIER_V4_MODELS_DIR / block
    compare_pairs.append({
        "block": block,
        "v3_test_macro_f1": infer_metric_value(v3_path / "test_metrics.csv", "macro_f1"),
        "v4_test_macro_f1": infer_metric_value(v4_path / "test_metrics.csv", "macro_f1"),
        "v3_test_accuracy": infer_metric_value(v3_path / "test_metrics.csv", "accuracy"),
        "v4_test_accuracy": infer_metric_value(v4_path / "test_metrics.csv", "accuracy"),
    })

for fam in ["web", "recon", "spoofing"]:
    v3_path = HIER_V3_MODELS_DIR / "level3_family_submodels" / fam
    v4_path = HIER_V4_MODELS_DIR / "level3_family_submodels" / fam
    compare_pairs.append({
        "block": fam,
        "v3_test_macro_f1": infer_metric_value(v3_path / "test_metrics.csv", "macro_f1"),
        "v4_test_macro_f1": infer_metric_value(v4_path / "test_metrics.csv", "macro_f1"),
        "v3_test_accuracy": infer_metric_value(v3_path / "test_metrics.csv", "accuracy"),
        "v4_test_accuracy": infer_metric_value(v4_path / "test_metrics.csv", "accuracy"),
    })

compare_pairs_df = pd.DataFrame(compare_pairs)
compare_pairs_df["gain_macro_f1"] = compare_pairs_df["v4_test_macro_f1"] - compare_pairs_df["v3_test_macro_f1"]
compare_pairs_df["gain_accuracy"] = compare_pairs_df["v4_test_accuracy"] - compare_pairs_df["v3_test_accuracy"]
compare_pairs_df


,block,v3_test_macro_f1,v4_test_macro_f1,v3_test_accuracy,v4_test_accuracy,gain_macro_f1,gain_accuracy
0,level1_binary,0.543347,0.696427,0.970259,0.960901,0.153080,-0.009358
1,level2_family,0.530652,0.555822,0.720007,0.744560,0.025170,0.024553
2,web,0.551673,0.587944,0.553080,0.594200,0.036271,0.041120
3,recon,0.450976,0.463275,0.474293,0.499022,0.012299,0.024729
4,spoofing,0.701311,0.681228,0.704956,0.684689,-0.020083,-0.020267


In [27]:
if len(compare_pairs_df) > 0:
    tmp = compare_pairs_df.copy()
    x = np.arange(len(tmp))
    width = 0.38

    plt.figure(figsize=(12, 5))
    plt.bar(x - width/2, tmp["v3_test_macro_f1"], width=width, label="V3")
    plt.bar(x + width/2, tmp["v4_test_macro_f1"], width=width, label="V4")
    plt.xticks(x, tmp["block"], rotation=25)
    plt.title("Comparaison V3 vs V4 — test macro-F1")
    plt.ylabel("Macro-F1")
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    save_fig("11_v3_vs_v4_macro_f1_blocks.png")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\11_v3_vs_v4_macro_f1_blocks.png


## 11) Tableau synthèse final : meilleurs résultats hiérarchique vs non hiérarchique

Cette section identifie :
- le meilleur résultat **non hiérarchique**
- le meilleur résultat **hiérarchique final**
- les paramètres associés
- les conclusions pratiques


In [28]:
# =========================
# 11) Best models summary
# =========================
flat_best = flat_summary_df.sort_values("test_macro_f1", ascending=False).head(1)
display(flat_best)

hier_best_block = hier_summary_df.sort_values("test_macro_f1", ascending=False).head(1)
display(hier_best_block)

if hier_final_summary is not None:
    final_pipeline_row = pd.DataFrame([{
        "name": "hierarchical_pipeline_final_v2",
        "category": "hierarchical_pipeline",
        "version": "v3_eval",
        "architecture": "Level1 + Level2 + Level3",
        "test_accuracy": hier_final_summary.get("final_metrics", {}).get("accuracy"),
        "test_macro_f1": hier_final_summary.get("final_metrics", {}).get("macro_f1"),
        "level1_accuracy": hier_final_summary.get("level1_metrics", {}).get("accuracy"),
        "level1_macro_f1": hier_final_summary.get("level1_metrics", {}).get("macro_f1"),
        "level2_accuracy": hier_final_summary.get("level2_metrics", {}).get("accuracy"),
        "level2_macro_f1": hier_final_summary.get("level2_metrics", {}).get("macro_f1"),
    }])
    display(final_pipeline_row)


,name,path,category,version,architecture,loss,best_epoch,best_val_macro_f1_from_summary,val_macro_f1,val_accuracy,test_macro_f1,test_accuracy
0,stable_flat_mlp_a,E:\dataset\processed_merged_full\archive_legac...,flat,pre_v3,32 -> 64 -> 34,None,11,0.562282,0.562282,0.744678,0.56349,0.745194


,name,path,category,version,architecture,loss,best_epoch,best_val_macro_f1_from_summary,val_macro_f1,val_accuracy,test_macro_f1,test_accuracy
4,hier_v3_level3_mirai,E:\dataset\processed_merged_full\hierarchical_...,hierarchical,v3,None,None,54,0.987334,0.987334,0.987341,0.987734,0.987741


,name,category,version,architecture,test_accuracy,test_macro_f1,level1_accuracy,level1_macro_f1,level2_accuracy,level2_macro_f1
0,hierarchical_pipeline_final_v2,hierarchical_pipeline,v3_eval,Level1 + Level2 + Level3,0.562963,0.536215,0.970275,0.543125,0.719136,0.471917


## 12) Analyse automatique et conclusion prête pour le rapport

Cette cellule produit un résumé textuel basé sur les artifacts présents.


In [29]:
# =========================
# 12) Résumé automatique
# =========================
best_flat_name = None
best_flat_f1 = None
if len(flat_summary_df) > 0 and flat_summary_df["test_macro_f1"].notna().any():
    best_flat_row = flat_summary_df.sort_values("test_macro_f1", ascending=False).iloc[0]
    best_flat_name = best_flat_row["name"]
    best_flat_f1 = best_flat_row["test_macro_f1"]

final_hier_f1 = None
final_hier_acc = None
if hier_final_summary is not None:
    final_hier_f1 = hier_final_summary.get("final_metrics", {}).get("macro_f1")
    final_hier_acc = hier_final_summary.get("final_metrics", {}).get("accuracy")

summary_lines = []
summary_lines.append("=== SYNTHÈSE AUTOMATIQUE ===")
summary_lines.append(f"Meilleur modèle non hiérarchique: {best_flat_name} | test macro-F1 = {best_flat_f1}")
summary_lines.append(f"Pipeline hiérarchique final: test macro-F1 = {final_hier_f1} | test accuracy = {final_hier_acc}")

if compare_pairs_df is not None and len(compare_pairs_df) > 0:
    improved_blocks = compare_pairs_df.sort_values('gain_macro_f1', ascending=False)[["block", "gain_macro_f1"]]
    summary_lines.append("Gains V4 vs V3 (macro-F1) sur blocs clés:")
    for _, row in improved_blocks.iterrows():
        summary_lines.append(f"  - {row['block']}: {row['gain_macro_f1']:.4f}")

print("\n".join(summary_lines))


=== SYNTHÈSE AUTOMATIQUE ===
Meilleur modèle non hiérarchique: stable_flat_mlp_a | test macro-F1 = 0.5634902557583477
Pipeline hiérarchique final: test macro-F1 = 0.5362145446146149 | test accuracy = 0.5629630307431172
Gains V4 vs V3 (macro-F1) sur blocs clés:
  - level1_binary: 0.1531
  - web: 0.0363
  - level2_family: 0.0252
  - recon: 0.0123
  - spoofing: -0.0201


## 13) Export des tableaux synthèse

Les fichiers suivants seront exportés dans `report_figures_pfe` :
- `flat_summary_table.csv`
- `hier_summary_table.csv`
- `v3_vs_v4_blocks.csv`


In [30]:
flat_summary_df.to_csv(FIG_DIR / "flat_summary_table.csv", index=False)
hier_summary_df.to_csv(FIG_DIR / "hier_summary_table.csv", index=False)
compare_pairs_df.to_csv(FIG_DIR / "v3_vs_v4_blocks.csv", index=False)

print("Saved:", FIG_DIR / "flat_summary_table.csv")
print("Saved:", FIG_DIR / "hier_summary_table.csv")
print("Saved:", FIG_DIR / "v3_vs_v4_blocks.csv")


Saved: E:\dataset\processed_merged_full\report_figures_pfe\flat_summary_table.csv
Saved: E:\dataset\processed_merged_full\report_figures_pfe\hier_summary_table.csv
Saved: E:\dataset\processed_merged_full\report_figures_pfe\v3_vs_v4_blocks.csv


# Notes d’interprétation pour le rapport

## Dataset
- La figure `01_raw_merged_shards_row_counts.png` documente le comportement des fichiers shard avant consolidation complète.
- Les figures `02_*` et `03_*` documentent la distribution après merge complet.
- La figure `05_*` montre l’effet direct du balancing / oversampling contrôlé.

## Preprocessing
- La figure `04_*` permet de justifier la suppression ou l’analyse des features constantes et corrélées.

## Expériences
- Les figures `08_*`, `09_*` et `11_*` comparent les performances de tous les modèles importants.
- Les figures `curve_*` montrent l’effet des changements de paramètres et la stabilité d’apprentissage.
- Le tableau `v3_vs_v4_blocks.csv` est particulièrement utile pour commenter l’impact de :
  - l’architecture plus profonde,
  - la focal loss,
  - le threshold tuning,
  - et la spécialisation des familles difficiles.

## Résultats finaux
- Le meilleur résultat **non hiérarchique** doit être pris dans `flat_summary_table.csv`.
- Le meilleur résultat **hiérarchique final** provient de `hierarchical_final_inference_v2/summary.json` et `level_metrics.csv`.
